# Settings

Import the libraries, packages and files required

In [2]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import json

In [3]:
DATA_DIR = Path("data")
GPKG_PATH = DATA_DIR / "snapshot.gpkg"

# 1. Attribute completeness

### Requirements

1. **Settings:** the required attributes list per class.
2. **Check each rule per feature:** one column per rule where True or False values are stored if the rule applies. For example has_name. Values like surface or religion are checked in the tags column, which holds several feature's tags.
3. **Feature score:** the share of rules it passes, from 0 to 1. A road with a name but no surface gets 0.5.
4. **Class score:** the average of its feature scores. This is the S for attribute completeness in the DQI in equation (1).
5. **Save the result:** a completeness table (feature_id, rule columns, score) and the errors (feature_id plus the rules it failed), so there's easy access or way to filter the failing features in QGIS.

### 1.1 Defining the attribute list and the rules

In [ ]:
MAIN_ROADS = ["motorway", "trunk", "primary", "secondary"]   # updated according the meeting 10/11/2026

PLACEHOLDERS = ["road", "fixme", "fix", "name", "updateme"]   # values that count as missing

RULES = {
    "roads": [
        {"rule": "has_name_or_ref", "tags": ["name", "ref"], "applies_to": {"highway": MAIN_ROADS}, "missing_values": PLACEHOLDERS},
        {"rule": "has_surface", "tags": ["surface"], "applies_to": {"highway": MAIN_ROADS}},
    ],
    "buildings": [
        {"rule": "has_building_type", "tags": ["building"], "applies_to": None, "missing_values": ["yes"]},
    ],
    "pois": [
        {"rule": "has_name", "tags": ["name"], "applies_to": None, "missing_values": PLACEHOLDERS},
        {"rule": "has_religion", "tags": ["religion"], "applies_to": {"amenity": ["place_of_worship"]}},
    ],
}

In [8]:
def check_rule(tags, rule):

    # 1. check if the rule apply to this feature
    conditions = rule["applies_to"]
    if conditions is not None:
        for key, allowed_values in conditions.items():
            feature_value = tags.get(key)
            if feature_value not in allowed_values:
                return None   # rule does not apply

    # 2. values that count as missing (if there are)
    if "missing_values" in rule:
        missing_values = rule["missing_values"]
    else:
        missing_values = []

    # 3. the rule passes if any of its tags has a useful value
    for tag_name in rule["tags"]:
        value = tags.get(tag_name)
        if value is None:
            continue   # tag not present, try the next one

        clean_value = value.strip().lower()   
        if clean_value == "":
            continue   # empty value (or only spaces), try the next one

        if clean_value in missing_values:
            continue   # placeholder, try the next one

        return True    # useful value found

    return False       # no tag had a useful value

#### 1.1.1 Roads

Checking the rules for roads

In [21]:
one_road = {"highway": "primary", "name": "FIX", "ref": "5"}
print(check_rule(one_road, RULES["roads"][0]))  

True


In [22]:
one_road = {"highway": "primary", "name": "FIX", "ref": "5", "surface":"a"} 
print(check_rule(one_road, RULES["roads"][1]))  

True


Check the rules to all of the roads

In [23]:
roads = gpd.read_file(GPKG_PATH, layer="roads")
tags = roads["tags"].apply(json.loads)

rule_cols = []

for rule in RULES["roads"]:
    results = []                          # one result per road
    for one_road_tags in tags:            # tags = the tags of every road
        result = check_rule(one_road_tags, rule)
        results.append(result)
    column_name = rule["rule"]            # e.g. "has_name_or_ref"
    roads[column_name] = pd.array(results, dtype="boolean")
    rule_cols.append(column_name)

# structure the passed and applicable rules and calculate the score per road
roads["passed"] = roads[rule_cols].sum(axis=1)
roads["applicable"] = roads[rule_cols].notna().sum(axis=1)
roads["score"] = roads["passed"] / roads["applicable"]
roads.head(3)

,feature_id,version,timestamp,user,highway,building,amenity,name,n_tags,tags,geometry,has_name_or_ref,has_surface,passed,applicable,score
0,w24930369,30,2026-06-20 17:56:28+00:00,Samer Aldahhak,secondary,None,None,اتوستراد السلام,4,"{""highway"": ""secondary"", ""is_in"": ""Syria"", ""na...","LINESTRING (35.91181 33.17095, 35.91282 33.172...",True,False,1,2,0.5
1,w24930371,18,2026-06-20 15:02:12+00:00,Samer Aldahhak,tertiary,None,None,NaN,1,"{""highway"": ""tertiary""}","LINESTRING (36.08232 33.20406, 36.07631 33.208...",<NA>,<NA>,0,0,<NA>
2,w24930382,2,2025-05-08 11:00:31+00:00,IknowJoseph,tertiary,None,None,NaN,1,"{""highway"": ""tertiary""}","LINESTRING (36.04078 33.15308, 36.04079 33.153...",<NA>,<NA>,0,0,<NA>


Show the summary of the rules assessed

In [24]:
for column_name in rule_cols:
    applied = roads[column_name].notna().sum()
    passed = roads[column_name].sum()
    if applied == 0:
        print(f"{column_name}: rule did not apply to any feature")
    else:
        share = passed / applied
        print(f"{column_name}: {passed} of {applied} pass ({share:.2%})")

scored_roads = roads[roads["applicable"] > 0]
print(f"Roads scored: {len(scored_roads)}")

has_name_or_ref: 86 of 203 pass (42.36%)
has_surface: 15 of 203 pass (7.39%)
Roads scored: 203


Calculating roads score by lenght

In [25]:
# 1. To get the result according the lenght (m, not degrees), suggested in the meeting 10/11/2026
roads_metric = roads.to_crs(roads.estimate_utm_crs())
roads["length_m"] = roads_metric.length

# 2. keep only roads where at least one rule applies
scored_roads = roads[roads["applicable"] > 0]

# 3. weighted score = sum(score x length) / sum(length)
weighted_sum = 0
total_length = 0
for index, road in scored_roads.iterrows():
    weighted_sum = weighted_sum + road["score"] * road["length_m"]
    total_length = total_length + road["length_m"]

class_score_roads = weighted_sum / total_length
print("Class score (roads, by count):", roads["score"].mean())
print("Class score (roads, by length):", class_score_roads)

Class score (roads, by count): 0.24876847290640394
Class score (roads, by length): 0.17373732984890145


### 1.1.2 Buildings

Check the rules for buildings

In [26]:
one_building = {"building": "church"}
print(check_rule(one_building, RULES["buildings"][0]))

True


Check the rules to all of the buildings

In [27]:
buildings = gpd.read_file(GPKG_PATH, layer="buildings")
building_tags = buildings["tags"].apply(json.loads)

building_rule_cols = []

for rule in RULES["buildings"]:
    results = []                                 
    for one_building_tags in building_tags:
        result = check_rule(one_building_tags, rule)
        results.append(result)

    column_name = rule["rule"]                    # e.g. "has_building_type"
    buildings[column_name] = pd.array(results, dtype="boolean")
    building_rule_cols.append(column_name)

# score per building: rules passed / rules that applied
buildings["passed"] = buildings[building_rule_cols].sum(axis=1)
buildings["applicable"] = buildings[building_rule_cols].notna().sum(axis=1)
buildings["score"] = buildings["passed"] / buildings["applicable"]


Summary per rule

In [28]:
for column_name in building_rule_cols:
    applied = buildings[column_name].notna().sum()
    passed = buildings[column_name].sum()
    if applied == 0:
        print(f"{column_name}: rule did not apply to any feature")
    else:
        share = passed / applied
        print(f"{column_name}: {passed} of {applied} pass ({share:.1%})")

has_building_type: 758 of 10432 pass (7.3%)


Calculating buildings score

In [29]:
# area in square metres (local metric projection)
buildings_metric = buildings.to_crs(buildings.estimate_utm_crs())
buildings["area_m2"] = buildings_metric.area

# area-weighted class score = sum(score x area) / sum(area)
scored_buildings = buildings[buildings["applicable"] > 0]
weighted_sum = 0
total_area = 0
for index, building in scored_buildings.iterrows():
    weighted_sum = weighted_sum + building["score"] * building["area_m2"]
    total_area = total_area + building["area_m2"]

class_score_buildings = weighted_sum / total_area
print("Class score (buildings, by count):", buildings["score"].mean())
print("Class score (buildings, by area):", class_score_buildings)

Class score (buildings, by count): 0.07266104294478527
Class score (buildings, by area): 0.11578713431138307


### 1.1.3 POIs

In [30]:
one_poi = {"name": "name"}
print(check_rule(one_poi, RULES["pois"][0]))

False


In [31]:
one_poi = {"religion": "c", "amenity": "place_of_worship"}
print(check_rule(one_poi, RULES["pois"][1]))


True


Check the rules to all of the points of interest

In [32]:
pois = gpd.read_file(GPKG_PATH, layer="pois")
poi_tags = pois["tags"].apply(json.loads)

poi_rule_cols = []

for rule in RULES["pois"]:
    results = []                                 
    for one_poi_tags in poi_tags:
        result = check_rule(one_poi_tags, rule)
        results.append(result)

    column_name = rule["rule"]                   
    pois[column_name] = pd.array(results, dtype="boolean")
    poi_rule_cols.append(column_name)

# score per poi: rules passed / rules that applied
pois["passed"] = pois[poi_rule_cols].sum(axis=1)
pois["applicable"] = pois[poi_rule_cols].notna().sum(axis=1)
pois["score"] = pois["passed"] / pois["applicable"]


Summary per rule

In [33]:
for column_name in poi_rule_cols:
    applied = pois[column_name].notna().sum()
    passed = pois[column_name].sum()
    if applied == 0:
        print(f"{column_name}: rule did not apply to any feature")
    else:
        share = passed / applied
        print(f"{column_name}: {passed} of {applied} pass ({share:.1%})")

has_name: 33 of 59 pass (55.9%)
has_religion: 39 of 45 pass (86.7%)


Calculating POIs score (based on amount of features)

In [34]:
class_score_pois = pois["score"].mean()
print("Class score (POIs, by count):", class_score_pois)


Class score (POIs, by count): 0.6949152542372882


## 1.2 Class score

In [35]:
# number of features that were actually scored in each class
n_roads = (roads["applicable"] > 0).sum()
n_buildings = (buildings["applicable"] > 0).sum()
n_pois = (pois["applicable"] > 0).sum()
n_total = n_roads + n_buildings + n_pois

attribute_completeness = (class_score_roads * n_roads + class_score_buildings * n_buildings + class_score_pois * n_pois) / n_total

print(f"Roads:     {class_score_roads:.3f}  ({n_roads} features)")
print(f"Buildings: {class_score_buildings:.3f}  ({n_buildings} features)")
print(f"POIs:      {class_score_pois:.3f}  ({n_pois} features)")
print(f"Attribute completeness: {attribute_completeness:.3f}")

Roads:     0.174  (203 features)
Buildings: 0.116  (10432 features)
POIs:      0.695  (59 features)
Attribute completeness: 0.120


# 2. Feature completeness